# LNR Stock Forecasting — One Agent, Three Tasks

> **Part 3 of 7.** This notebook builds on the agentic predictor introduced in
> [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb).

**Identity vs role.** One Analyst Agent (system prompt + toolbelt) answers three
different questions. The identity is fixed; only the **task spec** in the user
payload changes:

| Stream | Task | Output |
|--------|------|--------|
| 1 | Trajectory | 5/10/21-day price forecasts |
| 2 | Binary shock | P(LNR +$5 in 5 days) |
| 3 | Scenario analysis | Top 3 expert scenarios for 60 days |

A **task spec** is the ask: the question, the rules, and the required JSON shape.
It is *not* the system prompt. Edit the identity strings once, then edit each
stream's task spec and re-run (keep `USE_CACHE = False` after edits).


In [32]:
import random

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Random seed set to {SEED}")

Random seed set to 42


In [33]:
import sys
from pathlib import Path

ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "implementations").is_dir()),
    Path.cwd().resolve(),
)
sys.path[:0] = [str(ROOT / "aieng-forecasting"), str(ROOT / "implementations")]

import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display  # noqa: A004


warnings.filterwarnings("ignore")

# ── Model selection ───────────────────────────────────────────────────────────
# Two project models: "gemini-3.1-flash-lite-preview" (lite/default) and
# "gemini-3.5-flash" (advanced). Lite is the default here; switch to advanced
# for higher-quality runs.
AGENT_MODEL = "gemini-3.1-flash-lite-preview"

# ── Cache control ─────────────────────────────────────────────────────────────
# Set to False to force a full end-to-end agent run (ignores all cached results).
# Keep False if you edit the identity or any stream's task spec.
USE_CACHE = False

from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.methods.agentic import (
    AgentPredictor,
    ContinuousAgentForecastOutput,
    DiscreteAgentForecastOutput,
)
from energy_oil_for_lnr.analysis import compute_brier_score, trajectory_mae_table
from energy_oil_for_lnr.analyst_agent import build_lnr_multitask_news_config
from energy_oil_for_lnr.data import LNR_SERIES_ID, build_lnr_service, naive_utc_now
from energy_oil_for_lnr.paths import (
    PROPHET_SHOCK_TRAJ_CACHE,
    PROPHET_TRAJ_CACHE,
    SCENARIO_CACHE,
    SHOCK_ANALYST_CACHE,
    SHOCK_HORIZON,
    SHOCK_ORIGINS,
    SHOCK_THRESHOLD,
    TRAJ_AGENT_CACHE,
    TRAJECTORY_ORIGINS,
)
from energy_oil_for_lnr.prophet_baseline import (
    check_shock_outcome,
    load_prophet_trajectories,
    prophet_prob_shock,
    lnr_series_to_price_df,
)
from energy_oil_for_lnr.tasks import (
    ScenarioAgentForecastOutput,
    LnrMultitaskPromptBuilder,
)
from energy_oil_for_lnr.viz import (
    conf_bar,
    make_shock_comparison_chart,
    make_trajectory_fan_chart,
    prob_bar,
    verdict_label,
)


data_service = build_lnr_service()
ctx = data_service.context(as_of=naive_utc_now())
price_df = lnr_series_to_price_df(ctx.get_series(LNR_SERIES_ID))

prophet_traj_df = load_prophet_trajectories(price_df, TRAJECTORY_ORIGINS, PROPHET_TRAJ_CACHE)
prophet_shock_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(f"Price history through {price_df.index[-1].date()}")


def preview_user_payload(builder: LnrMultitaskPromptBuilder, task: ForecastingTask, origin: pd.Timestamp) -> None:
    """Show the JSON user payload the agent would receive (no model call)."""
    as_of = origin - pd.Timedelta(days=1)
    origin_ctx = data_service.context(as_of=as_of)
    payload = json.loads(builder(task=task, context=origin_ctx))
    hist_lines = payload["target_history_csv"].splitlines()
    ask_prose, _, ask_schema = payload["task_spec"].partition("Required JSON format:")
    display(
        Markdown(
            f"### User payload preview  "
            f"*(as_of {payload['as_of']}, LNR ${payload['origin_price_cad_per_share']:.2f}/share)*\n\n"
            "This is how we assign the task: the ask rides in `task_spec`; "
            "horizons and quantiles come from the `ForecastingTask`.\n\n"
            f"**Price history** — last 10 of {len(hist_lines) - 1} rows:\n\n"
            "```\n" + "\n".join(hist_lines[-10:]) + "\n```\n\n"
            f"**horizons:** `{payload['horizons']}`  ·  "
            f"**standard_quantiles:** {len(payload['standard_quantiles'])} levels\n\n"
            f"**task_spec** ({len(payload['task_spec'])} chars) — prose:\n\n"
            + ask_prose.strip()
            + "\n\n**Required JSON format:**\n\n```json\n"
            + ask_schema.strip()
            + "\n```"
        )
    )

Loaded 63 Prophet trajectory rows from lnr_energy_prophet_trajectories.parquet
Loaded 126 Prophet trajectory rows from lnr_energy_shock_prophet_trajectories.parquet
Price history through 2026-09-24


---
## Shared identity — system prompt + toolbelt

This is what the agent *is*. The same `analyst_config` is reused by all three streams.
Edit the strings below to change persona or search behaviour; do **not** put the
trajectory / shock / scenario ask here — that belongs in each stream's task spec.


In [34]:
# ── Editable identity (shared by Streams 1–3) ─────────────────────────────────
# Task-agnostic: persona + how to read the payload. The ask is NOT here.

SYSTEM_INSTRUCTION = """
## Role

You are an expert LNR LNR stock market analyst.

## Input

You will receive a JSON payload containing:
- `task_spec`: the exact question and required JSON output schema
- `as_of`: the forecast origin date (temporal cutoff)
- `horizons`: integer horizon steps (business days ahead)
- `standard_quantiles`: quantile levels for continuous forecasts (when applicable)
- `origin_price_cad_per_share`: LNR close on the origin date
- `target_history_csv`: compressed LNR daily close history

When context retrieval is enabled, call ``search_web`` BEFORE answering.

## Output contract

Read the data (and briefing, if retrieved) carefully, then execute the task in `task_spec` precisely.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response` — the exact schema is described in `task_spec`. Otherwise return the JSON directly as plain text with no preamble.
""".strip()

SEARCH_INSTRUCTION = """
You are an LNR stock-market intelligence specialist with access to web search.

Search for information relevant to the query and return a concise structured markdown summary (3-5 paragraphs) covering relevant aspects of:
- LNR stock price level and recent trend
- Linamar revenue, margins, and operating outlook
- Auto, industrial, and end-market demand conditions
- Management commentary, guidance, and capital allocation updates
- Key sector, customer, or competitor developments affecting LNR
- Published analyst views or notable price-target revisions
- Factors likely to impact future earnings, cash flow, or valuation
- North American vehicle production trends, EV adoption trends, and customer production outlooks
- New information that may not yet be fully reflected in LNR's share price
- Significant macroeconomic developments including interest rates, tariffs, commodity prices, exchange rates, and capital spending trends

Ground your summary in the search results you actually retrieve. When a cutoff date is specified, do not report or speculate about events that occurred after that date.

Before finalizing your summary, reason step by step: (1) for each candidate fact, judge its actual recency from the substance of the result itself, never from a source's claimed publish date or byline timestamp — those are frequently stale or updated after original publication; (2) discard anything you cannot confidently place before the cutoff date; (3) only then write your summary. Do not supplement the search results with your own background/training knowledge — if the results are insufficient, say so explicitly rather than filling gaps from memory.
""".strip()

_base = build_lnr_multitask_news_config(model=AGENT_MODEL)
analyst_config = _base.model_copy(
    update={
        "instruction": SYSTEM_INSTRUCTION,
        "context_retrieval": _base.context_retrieval.model_copy(update={"instruction": SEARCH_INSTRUCTION}),
    }
)

cr = analyst_config.context_retrieval
display(
    Markdown(
        f"### Toolbelt inventory  *(agent `{analyst_config.name}`, model `{analyst_config.model}`)*\n\n"
        "| Capability | Status |\n|---|---|\n"
        f"| `search_web` (context-retrieval sub-agent) | "
        f"{'**on**' if cr.enabled else 'off'} — cutoff = payload `as_of` |\n"
        f"| Search model | `{cr.search_model}` |\n"
        f"| Temporal-leakage verifier | `{cr.verifier_model}` "
        f"(max {cr.verifier_max_attempts} attempts, confidence ≥ {cr.verifier_confidence_threshold}) |\n"
        f"| Skills | none (`skills_dirs` empty) |\n"
        f"| Code execution | {'on' if analyst_config.code_execution.enabled else '**off**'} |\n"
        f"| `run_forecast` / function tools | "
        f"{'yes' if analyst_config.function_tools else '**none**'} |\n"
        "| `set_model_response` | attached **per stream** via `output_schema`, not by identity |\n"
    )
)
display(Markdown("### System instruction\n\n```\n" + SYSTEM_INSTRUCTION + "\n```"))
display(Markdown("### Search sub-agent instruction\n\n```\n" + SEARCH_INSTRUCTION + "\n```"))

### Toolbelt inventory  *(agent `lnr_analyst_multitask`, model `gemini-3.1-flash-lite-preview`)*

| Capability | Status |
|---|---|
| `search_web` (context-retrieval sub-agent) | **on** — cutoff = payload `as_of` |
| Search model | `gemini-3.1-flash-lite-preview` |
| Temporal-leakage verifier | `gemini-3.5-flash` (max 3 attempts, confidence ≥ 8) |
| Skills | none (`skills_dirs` empty) |
| Code execution | **off** |
| `run_forecast` / function tools | **none** |
| `set_model_response` | attached **per stream** via `output_schema`, not by identity |


### System instruction

```
## Role

You are an expert LNR LNR stock market analyst.

## Input

You will receive a JSON payload containing:
- `task_spec`: the exact question and required JSON output schema
- `as_of`: the forecast origin date (temporal cutoff)
- `horizons`: integer horizon steps (business days ahead)
- `standard_quantiles`: quantile levels for continuous forecasts (when applicable)
- `origin_price_cad_per_share`: LNR close on the origin date
- `target_history_csv`: compressed LNR daily close history

When context retrieval is enabled, call ``search_web`` BEFORE answering.

## Output contract

Read the data (and briefing, if retrieved) carefully, then execute the task in `task_spec` precisely.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response` — the exact schema is described in `task_spec`. Otherwise return the JSON directly as plain text with no preamble.
```

### Search sub-agent instruction

```
You are an LNR stock-market intelligence specialist with access to web search.

Search for information relevant to the query and return a concise structured markdown summary (3-5 paragraphs) covering relevant aspects of:
- LNR stock price level and recent trend
- Linamar revenue, margins, and operating outlook
- Auto, industrial, and end-market demand conditions
- Management commentary, guidance, and capital allocation updates
- Key sector, customer, or competitor developments affecting LNR
- Published analyst views or notable price-target revisions
- Factors likely to impact future earnings, cash flow, or valuation
- North American vehicle production trends, EV adoption trends, and customer production outlooks
- New information that may not yet be fully reflected in LNR's share price
- Significant macroeconomic developments including interest rates, tariffs, commodity prices, exchange rates, and capital spending trends

Ground your summary in the search results you actually retrieve. When a cutoff date is specified, do not report or speculate about events that occurred after that date.

Before finalizing your summary, reason step by step: (1) for each candidate fact, judge its actual recency from the substance of the result itself, never from a source's claimed publish date or byline timestamp — those are frequently stale or updated after original publication; (2) discard anything you cannot confidently place before the cutoff date; (3) only then write your summary. Do not supplement the search results with your own background/training knowledge — if the results are insufficient, say so explicitly rather than filling gaps from memory.
```

---
## Stream 1 — Trajectory Forecast

**Question:** Where will LNR be in 5, 10, and 21 business days?

Same identity as above. The task spec below is the ask — edit horizons or rules,
then re-run (keep `USE_CACHE = False`). Compare Prophet fan charts to the
news-grounded agent at three origins.

**Try this:** set `TRAJECTORY_HORIZONS = [5, 21]` and update the spec wording to match.


In [35]:
# ── Stream 1 task spec (edit this) ────────────────────────────────────────────
TRAJECTORY_HORIZONS = [5, 10, 21]  # feeds ForecastingTask; listed again in the ask

_TRAJ_SCHEMA = ContinuousAgentForecastOutput.prompt_schema_json()
TRAJECTORY_TASK_SPEC = f"""Forecast the LNR LNR stock price at each horizon listed in the payload
(`horizons`, business days ahead). Default horizons for this demo: {TRAJECTORY_HORIZONS}.

Rules:
  - Produce one forecast for each horizon in `horizons`.
  - Use exactly the quantile levels from `standard_quantiles` — no additions, no omissions.
  - `point_forecast` must exactly equal the 0.50 quantile value.
  - Quantile values must be strictly non-decreasing as quantile levels increase.
  - Document your reasoning in the `rationale` fields.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

Required JSON format:
{_TRAJ_SCHEMA}
"""

_prose, _, _schema = TRAJECTORY_TASK_SPEC.partition("Required JSON format:")
display(
    Markdown(
        "### Task spec — Stream 1\n\n"
        + _prose.strip()
        + "\n\n**Required JSON format** (`ContinuousAgentForecastOutput`):\n\n```json\n"
        + _schema.strip()
        + "\n```"
    )
)

### Task spec — Stream 1

Forecast the LNR LNR stock price at each horizon listed in the payload
(`horizons`, business days ahead). Default horizons for this demo: [5, 10, 21].

Rules:
  - Produce one forecast for each horizon in `horizons`.
  - Use exactly the quantile levels from `standard_quantiles` — no additions, no omissions.
  - `point_forecast` must exactly equal the 0.50 quantile value.
  - Quantile values must be strictly non-decreasing as quantile levels increase.
  - Document your reasoning in the `rationale` fields.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

**Required JSON format** (`ContinuousAgentForecastOutput`):

```json
{
  "forecasts": [
    {
      "horizon": "<integer \u2014 one entry per horizon from the task>",
      "point_forecast": "<float \u2014 must equal the 0.50 quantile value>",
      "quantiles": [
        {
          "quantile": 0.05,
          "value": "<float>"
        },
        {
          "quantile": 0.1,
          "value": "<float>"
        },
        {
          "quantile": 0.2,
          "value": "<float>"
        },
        {
          "quantile": 0.3,
          "value": "<float>"
        },
        {
          "quantile": 0.4,
          "value": "<float>"
        },
        {
          "quantile": 0.5,
          "value": "<float>"
        },
        {
          "quantile": 0.6,
          "value": "<float>"
        },
        {
          "quantile": 0.7,
          "value": "<float>"
        },
        {
          "quantile": 0.8,
          "value": "<float>"
        },
        {
          "quantile": 0.9,
          "value": "<float>"
        },
        {
          "quantile": 0.95,
          "value": "<float>"
        }
      ],
      "rationale": "<string>"
    }
  ],
  "rationale": "<string, optional overall explanation>"
}
```

In [36]:
# ── Assign role: wire identity + task spec (no model call) ────────────────────
trajectory_task = ForecastingTask(
    task_id="lnr_trajectory_demo",
    target_series_id=LNR_SERIES_ID,
    horizons=list(TRAJECTORY_HORIZONS),
    frequency="B",
    description="Trajectory demo for NB3",
)
traj_prompt_builder = LnrMultitaskPromptBuilder(task_spec=TRAJECTORY_TASK_SPEC)
traj_predictor = AgentPredictor(
    agent_config=analyst_config,
    prompt_builder=traj_prompt_builder,
    output_schema=ContinuousAgentForecastOutput,
)

print(f"Predictor schema: {traj_predictor.output_schema.__name__}")
preview_user_payload(traj_prompt_builder, trajectory_task, TRAJECTORY_ORIGINS[-1])

Predictor schema: ContinuousAgentForecastOutput


### User payload preview  *(as_of 2026-03-01, LNR $93.15/share)*

This is how we assign the task: the ask rides in `task_spec`; horizons and quantiles come from the `ForecastingTask`.

**Price history** — last 10 of 684 rows:

```
2026-02-13,94.04
2026-02-17,92.50
2026-02-18,91.40
2026-02-19,92.25
2026-02-20,92.67
2026-02-23,92.46
2026-02-24,94.07
2026-02-25,93.20
2026-02-26,93.80
2026-02-27,93.15
```

**horizons:** `[5, 10, 21]`  ·  **standard_quantiles:** 11 levels

**task_spec** (1832 chars) — prose:

Forecast the LNR LNR stock price at each horizon listed in the payload
(`horizons`, business days ahead). Default horizons for this demo: [5, 10, 21].

Rules:
  - Produce one forecast for each horizon in `horizons`.
  - Use exactly the quantile levels from `standard_quantiles` — no additions, no omissions.
  - `point_forecast` must exactly equal the 0.50 quantile value.
  - Quantile values must be strictly non-decreasing as quantile levels increase.
  - Document your reasoning in the `rationale` fields.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

**Required JSON format:**

```json
{
  "forecasts": [
    {
      "horizon": "<integer \u2014 one entry per horizon from the task>",
      "point_forecast": "<float \u2014 must equal the 0.50 quantile value>",
      "quantiles": [
        {
          "quantile": 0.05,
          "value": "<float>"
        },
        {
          "quantile": 0.1,
          "value": "<float>"
        },
        {
          "quantile": 0.2,
          "value": "<float>"
        },
        {
          "quantile": 0.3,
          "value": "<float>"
        },
        {
          "quantile": 0.4,
          "value": "<float>"
        },
        {
          "quantile": 0.5,
          "value": "<float>"
        },
        {
          "quantile": 0.6,
          "value": "<float>"
        },
        {
          "quantile": 0.7,
          "value": "<float>"
        },
        {
          "quantile": 0.8,
          "value": "<float>"
        },
        {
          "quantile": 0.9,
          "value": "<float>"
        },
        {
          "quantile": 0.95,
          "value": "<float>"
        }
      ],
      "rationale": "<string>"
    }
  ],
  "rationale": "<string, optional overall explanation>"
}
```

In [40]:
# ── Run trajectory agent at three origins ─────────────────────────────────────
# Uses analyst_config + TRAJECTORY_TASK_SPEC. Keep USE_CACHE = False after edits.
if USE_CACHE and TRAJ_AGENT_CACHE.exists():
    with open(TRAJ_AGENT_CACHE) as f:
        traj_agent_results = json.load(f)
    print(f"Loaded {len(traj_agent_results)} cached trajectory agent runs.")
else:
    traj_agent_results = []
    for origin in TRAJECTORY_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        try:
            preds = traj_predictor.predict(trajectory_task, origin_ctx)
        except Exception as exc:  # pragma: no cover - provider/quota failure is external to code
            print(
                "Trajectory forecast failed because the model provider rejected the request. "
                "This is usually caused by exhausted API quota or budget, not a notebook bug."
            )
            print(f"{type(exc).__name__}: {exc}")
            break
        traj_agent_results.append(
            {
                "origin": str(origin.date()),
                "predictions": [p.model_dump(mode="json") for p in preds],
            }
        )
    if traj_agent_results:
        with open(TRAJ_AGENT_CACHE, "w") as f:
            json.dump(traj_agent_results, f, indent=2)
        print(f"Saved {len(traj_agent_results)} agent trajectory runs.")
    else:
        print("No trajectory results were generated; check model availability / project budget.")

print("\nAgent trajectory summary:")
for r in traj_agent_results:
    preds = r["predictions"]
    hs = TRAJECTORY_HORIZONS
    pts = [f"h{hs[i]}=${preds[i]['payload']['point_forecast']:.1f}" for i in range(len(preds))]
    origin_price_rows = price_df[price_df.index >= pd.Timestamp(r["origin"])]
    origin_price = f"LNR=${origin_price_rows.iloc[0]['price']:.2f}" if not origin_price_rows.empty else ""
    print(f"  {r['origin']}  {origin_price}  {' | '.join(pts)}")


Saved 3 agent trajectory runs.

Agent trajectory summary:
  2026-02-02  LNR=$85.57  h5=$85.8 | h10=$86.6 | h21=$87.8
  2026-02-23  LNR=$92.46  h5=$93.8 | h10=$95.2 | h21=$97.5
  2026-03-02  LNR=$93.83  h5=$93.7 | h10=$94.2 | h21=$95.1


In [41]:
# ── I/O inspection: 2026-03-02 — conflict onset, most informative ────────────
INSPECT_ORIGIN = "2026-03-02"
inspect_rec = next((r for r in traj_agent_results if r["origin"] == INSPECT_ORIGIN), None)

if inspect_rec:
    origin_ts = pd.Timestamp(INSPECT_ORIGIN)
    bday_dates = pd.bdate_range(start=origin_ts + pd.offsets.BDay(1), periods=max(TRAJECTORY_HORIZONS))
    origin_price_row = price_df[price_df.index >= origin_ts]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")

    preds = inspect_rec["predictions"]
    rationale = preds[0].get("metadata", {}).get("rationale", "") if preds else ""

    table_rows = "| Horizon | Agent ($) | 80% CI | Actual ($) | Agent err | Prophet err |\n|---|---|---|---|---|---|\n"
    for i, h in enumerate(TRAJECTORY_HORIZONS):
        actual_rows = price_df[price_df.index >= bday_dates[h - 1]]
        actual = float(actual_rows.iloc[0]["price"]) if not actual_rows.empty else float("nan")
        pt = preds[i]["payload"]["point_forecast"]
        q10_val = next(
            (v for k, v in preds[i]["payload"]["quantiles"].items() if abs(float(k) - 0.1) < 1e-6), float("nan")
        )
        q90_val = next(
            (v for k, v in preds[i]["payload"]["quantiles"].items() if abs(float(k) - 0.9) < 1e-6), float("nan")
        )
        p_row = prophet_traj_df[(prophet_traj_df["origin"] == origin_ts) & (prophet_traj_df["horizon"] == h)]
        p_yhat = float(p_row.iloc[0]["yhat"]) if not p_row.empty else float("nan")
        table_rows += (
            f"| {h} bdays | **${pt:.1f}** | [{q10_val:.1f} – {q90_val:.1f}] "
            f"| ${actual:.1f} | {pt - actual:+.1f} | {p_yhat - actual:+.1f} |\n"
        )

    display(
        Markdown(
            f"### Stream 1 — I/O Inspection: {INSPECT_ORIGIN}  (LNR ${origin_price:.2f}/share)\n\n"
            "Agent and Prophet point forecasts vs realised prices at each horizon.\n\n"
            + table_rows
            + (f"\n> **Agent rationale:** {rationale}" if rationale else "")
        )
    )

### Stream 1 — I/O Inspection: 2026-03-02  (LNR $93.83/share)

Agent and Prophet point forecasts vs realised prices at each horizon.

| Horizon | Agent ($) | 80% CI | Actual ($) | Agent err | Prophet err |
|---|---|---|---|---|---|
| 5 bdays | **$93.7** | [92.0 – 95.5] | $86.4 | +7.2 | -15.2 |
| 10 bdays | **$94.2** | [91.2 – 97.1] | $86.1 | +8.1 | -16.1 |
| 21 bdays | **$95.1** | [90.0 – 100.5] | $85.7 | +9.4 | -16.1 |

> **Agent rationale:** The forecasts are based on the strong upward price trend observed in LNR's closing data throughout the first two months of 2026. The stock is currently trading near historical highs, leading to a bullish central forecast, with quantiles appropriately widened to reflect increased uncertainty over longer horizons.

In [20]:
# ── Trajectory fan chart: Prophet fan vs agent error bars at 3 origins ───────
fig = make_trajectory_fan_chart(traj_agent_results, prophet_traj_df, price_df, TRAJECTORY_ORIGINS)
fig.show()

# ── MAE evaluation table ──────────────────────────────────────────────────────
mae_df = trajectory_mae_table(traj_agent_results, prophet_traj_df, price_df)
if not mae_df.empty:
    display(mae_df.drop(columns=["Prophet MAE", "Agent MAE"]))
    mean_mae = mae_df[["Prophet MAE", "Agent MAE"]].mean()
    print(f"\nMean MAE  Prophet: ${mean_mae['Prophet MAE']:.2f}  Agent: ${mean_mae['Agent MAE']:.2f}")

Actual ($) Prophet ($) Agent ($)
Origin     Horizon                                  
2026-02-02 5 bdays        87.9        68.6      85.8
           10 bdays       92.5        68.5      86.6
           21 bdays       91.9        68.4      88.2
2026-02-23 5 bdays        93.8        70.9      93.1
           10 bdays       86.4        69.9      94.5
           21 bdays       84.2        69.1      97.0
2026-03-02 5 bdays        86.4        71.2      93.7
           10 bdays       86.1        70.0      94.8
           21 bdays       85.7        69.6      96.2


Mean MAE  Prophet: $18.75  Agent: $6.63


---
## Stream 2 — Binary Shock Prediction

**Question:** What is P(LNR closes more than $5/share higher in 5 trading days)?

Same identity. A different task spec. Edit the threshold or horizon wording below —
if you change the scored definition, also update `SHOCK_THRESHOLD` / `SHOCK_HORIZON`
so the scorer stays aligned.

**Try this:** raise the bar to +$10 and compare probabilities.


In [8]:
# ── Stream 2 task spec (edit this) ────────────────────────────────────────────
# Scorer uses SHOCK_THRESHOLD / SHOCK_HORIZON from paths.py — keep them in sync.
_SHOCK_SCHEMA = DiscreteAgentForecastOutput.prompt_schema_json()
SHOCK_TASK_SPEC = f"""Estimate P(up) — the probability that LNR will close MORE THAN
${int(SHOCK_THRESHOLD)}/share HIGHER than today's price at the end of
{SHOCK_HORIZON} trading days.

This is a directional upside question only.

Calibration guidance:
  - No unusual upside catalyst       -> base rate ~10-15%
  - Escalating unconfirmed risk      -> 20-40%
  - Confirmed supply disruption      -> 60-85%

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

Required JSON format:
{_SHOCK_SCHEMA}
"""

_prose, _, _schema = SHOCK_TASK_SPEC.partition("Required JSON format:")
display(
    Markdown(
        "### Task spec — Stream 2\n\n"
        + _prose.strip()
        + "\n\n**Required JSON format** (`DiscreteAgentForecastOutput`):\n\n```json\n"
        + _schema.strip()
        + "\n```"
    )
)

### Task spec — Stream 2

Estimate P(up) — the probability that LNR will close MORE THAN
$5/share HIGHER than today's price at the end of
5 trading days.

This is a directional upside question only.

Calibration guidance:
  - No unusual upside catalyst       -> base rate ~10-15%
  - Escalating unconfirmed risk      -> 20-40%
  - Confirmed supply disruption      -> 60-85%

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

**Required JSON format** (`DiscreteAgentForecastOutput`):

```json
{
  "probability": "<float in [0, 1]>",
  "direction_bias": "<'up' | 'down' | 'neutral'>",
  "reasoning": "<string>",
  "key_signals": [
    "<signal 1>",
    "<signal 2>"
  ],
  "confidence": "<'high' | 'medium' | 'low'>"
}
```

In [9]:
# ── Assign role: wire identity + task spec (no model call) ────────────────────
shock_task = ForecastingTask(
    task_id="lnr_upshock_demo",
    target_series_id=LNR_SERIES_ID,
    horizons=[SHOCK_HORIZON],
    frequency="B",
    description="Binary upshock demo",
)
shock_prompt_builder = LnrMultitaskPromptBuilder(task_spec=SHOCK_TASK_SPEC)
shock_predictor = AgentPredictor(
    agent_config=analyst_config,
    prompt_builder=shock_prompt_builder,
    output_schema=DiscreteAgentForecastOutput,
)

print(f"Predictor schema: {shock_predictor.output_schema.__name__}")
preview_user_payload(shock_prompt_builder, shock_task, SHOCK_ORIGINS[-1])

Predictor schema: DiscreteAgentForecastOutput


### User payload preview  *(as_of 2026-03-08, LNR $87.63/share)*

This is how we assign the task: the ask rides in `task_spec`; horizons and quantiles come from the `ForecastingTask`.

**Price history** — last 10 of 683 rows:

```
2026-02-23,92.46
2026-02-24,94.07
2026-02-25,93.20
2026-02-26,93.80
2026-02-27,93.15
2026-03-02,93.83
2026-03-03,91.90
2026-03-04,91.83
2026-03-05,94.32
2026-03-06,87.63
```

**horizons:** `[5]`  ·  **standard_quantiles:** 11 levels

**task_spec** (744 chars) — prose:

Estimate P(up) — the probability that LNR will close MORE THAN
$5/share HIGHER than today's price at the end of
5 trading days.

This is a directional upside question only.

Calibration guidance:
  - No unusual upside catalyst       -> base rate ~10-15%
  - Escalating unconfirmed risk      -> 20-40%
  - Confirmed supply disruption      -> 60-85%

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

**Required JSON format:**

```json
{
  "probability": "<float in [0, 1]>",
  "direction_bias": "<'up' | 'down' | 'neutral'>",
  "reasoning": "<string>",
  "key_signals": [
    "<signal 1>",
    "<signal 2>"
  ],
  "confidence": "<'high' | 'medium' | 'low'>"
}
```

In [10]:
# ── Run shock agent across origins ────────────────────────────────────────────
# Uses analyst_config + SHOCK_TASK_SPEC. Keep USE_CACHE = False after edits.
if USE_CACHE and SHOCK_ANALYST_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_results = json.load(f)
    print(f"Loaded {len(shock_results)} cached shock forecasts.")
else:
    shock_results = []
    for origin in SHOCK_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = shock_predictor.predict(shock_task, origin_ctx)
        outcome, delta = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
        shock_results.append(
            {
                "origin": str(origin.date()),
                "probability": preds[0].payload.probability,
                "outcome": outcome,
                "delta": delta,
                "metadata": preds[0].metadata,
            }
        )
    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_results, f, indent=2)
    print(f"Saved {len(shock_results)} shock forecasts.")

agent_probs = [r["probability"] for r in shock_results]
outcomes = [r["outcome"] for r in shock_results]
print(f"Agent Brier score: {compute_brier_score(agent_probs, outcomes):.4f}")

Node execution failed with exception
Traceback (most recent call last):
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/llms/openai/openai.py", line 930, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/llms/openai/openai.py", line 461, in make_openai_chat_completion_request
    raise e
  File "/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/litellm/llms/openai/openai.py", line 438, in make_openai_chat_completion_request
    await openai_aclient.chat.completions.with_raw_response.create(
  File "/home/coder/agentic-fo


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ServiceUnavailableError: litellm.ServiceUnavailableError: ServiceUnavailableError: OpenAIException - Error code: 503 - {'detail': '{\n  "error": {\n    "code": 503,\n    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",\n    "status": "UNAVAILABLE"\n  }\n}\n'}

In [ ]:
# ── Per-origin forecast cards ─────────────────────────────────────────────────
for r in shock_results:
    origin = pd.Timestamp(r["origin"])
    label = origin.strftime("%b %-d, %Y")
    origin_price_row = price_df[price_df.index >= origin]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")
    a_prob = float(r["probability"])
    outcome = int(r["outcome"])
    delta = float(r["delta"])
    brier = (a_prob - outcome) ** 2
    meta = r.get("metadata", {})
    reasoning = meta.get("rationale", "—")
    key_signals = meta.get("key_signals", [])
    confidence = meta.get("confidence", "?")
    outcome_badge = "**SHOCK**" if outcome else "No shock"

    display(
        Markdown(
            f"---\n"
            f"### {label} — LNR ${origin_price:.2f}/share\n\n"
            f"| | |\n|---|---|\n"
            f"| **Prediction** | P(up > +${SHOCK_THRESHOLD:.0f}) = **{a_prob:.0%}**  `{prob_bar(a_prob)}` |\n"
            f"| **Confidence** | {confidence.title() if isinstance(confidence, str) else confidence}  {conf_bar(str(confidence))} |\n"
            f"| **Rationale** | {reasoning} |\n"
            f"| **Key signals** | {' · '.join(key_signals) if key_signals else '—'} |\n"
            f"| **Actual outcome** | {outcome_badge} — price moved **{delta:+.2f}/share** |\n"
            f"| **Verdict** | {verdict_label(a_prob, outcome, delta, SHOCK_THRESHOLD)} |\n"
            f"| **Brier score** | {brier:.3f} {'🟢' if brier < 0.10 else '🟡' if brier < 0.25 else '🔴'} |\n"
        )
    )

---
### Feb 2, 2026 — LNR $85.57/share

| | |
|---|---|
| **Prediction** | P(up > +$5) = **15%**  `██░░░░░░░░  15%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | LNR has been exhibiting relative price stability, consolidating in the $85-$87 range throughout January 2026. A $5 move (an approximate 5.8% increase) within 5 trading days would represent a significant deviation from recent volatility patterns, which have been relatively contained. There are no confirmed catalysts (e.g., earnings releases or major supply chain disruptions) identified as of Feb 1, 2026, to justify a high-probability breakout of this magnitude in the short term. The stock is currently trading near its recent highs, and the base rate for such an 'upshock' in stable conditions is appropriate. |
| **Key signals** | Stock consolidation between $85 and $87 during late January · Lack of immediate high-impact catalysts or market-moving news · Current price near local resistance levels |
| **Actual outcome** | No shock — price moved **+2.31/share** |
| **Verdict** | Actual: +$2.31/bbl — no shock |
| **Brier score** | 0.022 🟢 |


---
### Feb 9, 2026 — LNR $87.87/share

| | |
|---|---|
| **Prediction** | P(up > +$5) = **12%**  `█░░░░░░░░░  12%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | LNR is currently trading at $87.77, near its recent range highs. While the company has shown operational resilience, there are no specific upcoming catalysts identified (like an earnings report or a major contract win) that suggest an imminent ~5.7% upward breakout within the next 5 trading days. The market sentiment remains cautious regarding industrial headwinds and tariff risks. A 12% probability accounts for the possibility of general market volatility or a positive sector-wide move, but the base rate for an 'upshock' of this magnitude without a specific catalyst is low. |
| **Key signals** | Stock is trading near recent highs, suggesting resistance · Lack of immediate positive catalysts for a sharp breakout · Persistent macro and trade-related headwinds for the industrial/mobility sector · Historical volatility does not indicate a high frequency of 5%+ moves within a single week |
| **Actual outcome** | No shock — price moved **+4.63/share** |
| **Verdict** | Actual: +$4.63/bbl — no shock |
| **Brier score** | 0.014 🟢 |


---
### Feb 16, 2026 — LNR $92.50/share

| | |
|---|---|
| **Prediction** | P(up > +$5) = **18%**  `██░░░░░░░░  18%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | LNR experienced a significant price jump on Feb 13, 2026, closing at 94.04 CAD (up nearly 5.5% from the previous day's 89.15). While this momentum is bullish, the task requires a move of MORE THAN $5 (i.e., > 99.04 CAD) within 5 trading days. Such a rapid sustained rally following a major breakout is statistically less common without a new, specific material catalyst (e.g., earnings surprise or major deal announcement). The current environment remains focused on long-term fundamentals and navigating existing macroeconomic headwinds. Therefore, while the momentum is strong, the probability of extending this specific gain by another $5+ is assessed slightly above the base rate but well below event-driven disruption levels. |
| **Key signals** | Significant single-day breakout on Feb 13, 2026 (+5.48%) · Positive, stable, but not 'disruptive' macroeconomic outlook · Historical tendency for mean reversion after large short-term volatility spikes |
| **Actual outcome** | No shock — price moved **-0.04/share** |
| **Verdict** | Actual: +$-0.04/bbl — no shock |
| **Brier score** | 0.032 🟢 |


---
### Feb 23, 2026 — LNR $92.46/share

| | |
|---|---|
| **Prediction** | P(up > +$5) = **15%**  `██░░░░░░░░  15%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | LNR has been trading in a relatively stable range between $85 and $94 recently. While the stock has seen some volatility, there is no evidence of a specific, imminent catalyst (like a major contract win or unexpected supply chain collapse) that would drive a $5+ (roughly 5.4%) move in just 5 trading days. The market environment is characterized by cautious optimism tempered by macro concerns, suggesting that while the stock is resilient, a significant, rapid upside breakout is less likely than continued range-bound behavior. |
| **Key signals** | Stable trading range near $92.67 · Absence of immediate positive catalysts · Mixed macroeconomic environment (potential tariff/cyclical risks) |
| **Actual outcome** | No shock — price moved **+1.37/share** |
| **Verdict** | Actual: +$1.37/bbl — no shock |
| **Brier score** | 0.022 🟢 |


---
### Mar 2, 2026 — LNR $93.83/share

| | |
|---|---|
| **Prediction** | P(up > +$5) = **15%**  `██░░░░░░░░  15%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | LNR is scheduled to release its Q4 and full-year 2025 earnings on March 4, 2026. While earnings reports can act as catalysts for significant price moves, the current market context does not suggest an unusual upside catalyst beyond typical earnings volatility. The stock has been trading in a relatively tight range recently, and historical patterns suggest that absent a significant earnings surprise, a $5 move (approx. 5.4%) in 5 trading days is above average but within the realm of volatility expected around earnings. A 15% probability reflects the 'base rate' scenario, acknowledging the binary risk of the upcoming earnings release. |
| **Key signals** | Upcoming Q4/Full-Year 2025 earnings release on March 4, 2026 · Current stock stability within a mature range · Balanced analyst sentiment heading into the earnings report |
| **Actual outcome** | No shock — price moved **-7.42/share** |
| **Verdict** | Actual: +$-7.42/bbl — no shock |
| **Brier score** | 0.022 🟢 |


---
### Mar 9, 2026 — LNR $86.41/share

| | |
|---|---|
| **Prediction** | P(up > +$5) = **15%**  `██░░░░░░░░  15%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | LNR recently experienced a sharp correction on March 6, 2026, dropping significantly from its March 5 high of $94.32 to close at $87.63. While the company's long-term fundamentals remain supported by strong 2025 earnings and a robust mobility segment, the stock is currently digesting this sudden sell-off. Achieving a $5/share increase (reaching $92.63+) within 5 trading days would require a swift recovery to levels seen just days ago. Given the base rate for such a move without a specific new catalyst is low, and the stock is technically oversold after the recent drop, the probability of a rebound is higher than a typical downtrend but not a baseline expectation of a sustained breakout in the short term. |
| **Key signals** | Recent significant price correction on March 6, 2026 · Strong 2025 financial performance provides a fundamental floor · Technical recovery potential after sharp pullback |
| **Actual outcome** | No shock — price moved **-0.31/share** |
| **Verdict** | Actual: +$-0.31/bbl — no shock |
| **Brier score** | 0.022 🟢 |


In [ ]:
# ── Prophet probabilities for the shock origins ───────────────────────────────
prophet_shock_probs = []
for r in shock_results:
    origin = pd.Timestamp(r["origin"])
    origin_price_row = price_df[price_df.index >= origin]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")
    p_sub = prophet_shock_df[prophet_shock_df["origin"] == origin]
    prophet_shock_probs.append(prophet_prob_shock(p_sub, origin_price, SHOCK_THRESHOLD, SHOCK_HORIZON))

# ── Comparison chart: P(shock) over time + cumulative Brier ──────────────────
fig = make_shock_comparison_chart(shock_results, prophet_shock_probs, shock_threshold=SHOCK_THRESHOLD)
fig.show()

# ── Brier score summary ───────────────────────────────────────────────────────
agent_probs = [float(r["probability"]) for r in shock_results]
outcomes = [int(r["outcome"]) for r in shock_results]
agent_brier = compute_brier_score(agent_probs, outcomes)
valid_prophet = [(p, o) for p, o in zip(prophet_shock_probs, outcomes) if not np.isnan(p)]
prophet_brier = compute_brier_score([p for p, _ in valid_prophet], [o for _, o in valid_prophet])
brier_df = pd.DataFrame(
    {"Mean Brier score": [f"{agent_brier:.4f}", f"{prophet_brier:.4f}"]},
    index=pd.Index(["Analyst Agent", "Prophet"], name="Method"),
)
print("Mean Brier score (lower = better, 0.25 = random ceiling):")
display(brier_df)

Mean Brier score (lower = better, 0.25 = random ceiling):


,Mean Brier score
Method,
Analyst Agent,0.0228
Prophet,0.0000


---
## Stream 3 — Scenario Analysis

**Question:** What three scenarios are stock-market analysts and industry experts debating for LNR over the next 60 days?

Same identity. Track 2 structured qualitative analysis — no ground truth to score.
Edit the task spec (number of scenarios, framing) or the origin, then re-run.

**Try this:** change "three scenarios" to "two bullish and one bearish", or set
`SCENARIO_AS_OF = pd.Timestamp("2026-02-02")` (pre-shock) and compare.


In [ ]:
# ── Stream 3 task spec (edit this) ────────────────────────────────────────────
# SCENARIO_AS_OF = SCENARIO_ORIGIN  # 2026-03-02 — conflict onset
SCENARIO_AS_OF = pd.Timestamp("2026-02-02")  # pre-shock, quieter market
# SCENARIO_AS_OF = pd.Timestamp.today()  # live — no deep historical fence

_SCENARIO_SCHEMA = ScenarioAgentForecastOutput.prompt_schema_json()
SCENARIO_TASK_SPEC = f"""Identify the three scenarios that market analysts and industry experts are most
actively debating for LNR.TO stock over the next 60 days, given the current
market context and price history.

For each scenario:
  - Give it a concise name (3-6 words)
  - Describe it in 1-2 sentences
  - Assign a probability (all three must sum to <= 1.0)
  - Provide an expected LNR price range at the 60-day horizon as [low, high]
  - Give your point estimate for LNR at 60 days under this scenario
  - List 1-2 key drivers that would cause this scenario to materialise

Also identify which scenario is the base case and provide an overall
one-paragraph reasoning summary.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

Required JSON format:
{_SCENARIO_SCHEMA}
"""

_origin_price_row = price_df[price_df.index >= SCENARIO_AS_OF]
_origin_price = float(_origin_price_row.iloc[0]["price"]) if not _origin_price_row.empty else float("nan")
_prose, _, _schema = SCENARIO_TASK_SPEC.partition("Required JSON format:")
display(
    Markdown(
        f"### Task spec — Stream 3  *(origin {SCENARIO_AS_OF.date()}, LNR ${_origin_price:.2f}/share)*\n\n"
        + _prose.strip()
        + "\n\n**Required JSON format** (`ScenarioAgentForecastOutput`):\n\n```json\n"
        + _schema.strip()
        + "\n```"
    )
)

### Task spec — Stream 3  *(origin 2026-02-02, LNR $85.57/share)*

Identify the three scenarios that market analysts and industry experts are most
actively debating for LNR.TO stock over the next 60 days, given the current
market context and price history.

For each scenario:
  - Give it a concise name (3-6 words)
  - Describe it in 1-2 sentences
  - Assign a probability (all three must sum to <= 1.0)
  - Provide an expected LNR price range at the 60-day horizon as [low, high]
  - Give your point estimate for LNR at 60 days under this scenario
  - List 1-2 key drivers that would cause this scenario to materialise

Also identify which scenario is the base case and provide an overall
one-paragraph reasoning summary.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

**Required JSON format** (`ScenarioAgentForecastOutput`):

```json
{
  "scenarios": [
    {
      "name": "<string>",
      "description": "<string>",
      "probability": "<float in [0, 1]>",
      "lnr_range_60d": [
        "<float_low>",
        "<float_high>"
      ],
      "point_estimate_60d": "<float>",
      "key_drivers": [
        "<driver 1>",
        "<driver 2>"
      ]
    }
  ],
  "base_case": "<scenario name>",
  "reasoning": "<paragraph>"
}
```

In [ ]:
# ── Assign role: wire identity + task spec (no model call) ────────────────────
scenario_task = ForecastingTask(
    task_id="lnr_scenario_demo",
    target_series_id=LNR_SERIES_ID,
    horizons=[21],  # ForecastingTask requires a horizon; the 60-day ask lives in the spec
    frequency="B",
    description="Scenario analysis demo",
)
scenario_prompt_builder = LnrMultitaskPromptBuilder(task_spec=SCENARIO_TASK_SPEC)
scenario_predictor = AgentPredictor(
    agent_config=analyst_config,
    prompt_builder=scenario_prompt_builder,
    output_schema=ScenarioAgentForecastOutput,
)

print(f"Predictor schema: {scenario_predictor.output_schema.__name__}")
preview_user_payload(scenario_prompt_builder, scenario_task, SCENARIO_AS_OF)

Predictor schema: ScenarioAgentForecastOutput


### User payload preview  *(as_of 2026-02-01, LNR $85.53/share)*

This is how we assign the task: the ask rides in `task_spec`; horizons and quantiles come from the `ForecastingTask`.

**Price history** — last 10 of 680 rows:

```
2026-01-19,87.18
2026-01-20,86.41
2026-01-21,88.57
2026-01-22,88.42
2026-01-23,86.62
2026-01-26,87.61
2026-01-27,87.40
2026-01-28,86.32
2026-01-29,85.55
2026-01-30,85.53
```

**horizons:** `[21]`  ·  **standard_quantiles:** 11 levels

**task_spec** (1223 chars) — prose:

Identify the three scenarios that market analysts and industry experts are most
actively debating for LNR.TO stock over the next 60 days, given the current
market context and price history.

For each scenario:
  - Give it a concise name (3-6 words)
  - Describe it in 1-2 sentences
  - Assign a probability (all three must sum to <= 1.0)
  - Provide an expected LNR price range at the 60-day horizon as [low, high]
  - Give your point estimate for LNR at 60 days under this scenario
  - List 1-2 key drivers that would cause this scenario to materialise

Also identify which scenario is the base case and provide an overall
one-paragraph reasoning summary.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`. Otherwise return the JSON directly as plain text.

**Required JSON format:**

```json
{
  "scenarios": [
    {
      "name": "<string>",
      "description": "<string>",
      "probability": "<float in [0, 1]>",
      "lnr_range_60d": [
        "<float_low>",
        "<float_high>"
      ],
      "point_estimate_60d": "<float>",
      "key_drivers": [
        "<driver 1>",
        "<driver 2>"
      ]
    }
  ],
  "base_case": "<scenario name>",
  "reasoning": "<paragraph>"
}
```

In [ ]:
# ── Run the scenario agent ────────────────────────────────────────────────────
# Uses analyst_config + SCENARIO_TASK_SPEC. Keep USE_CACHE = False after edits.
if USE_CACHE:
    print("USE_CACHE is True — cached cards ignore edits to the task spec.")

if USE_CACHE and SCENARIO_CACHE.exists():
    with open(SCENARIO_CACHE) as f:
        scenario_payload = json.load(f)
    print("Loaded cached scenario analysis.")
else:
    as_of = SCENARIO_AS_OF - pd.Timedelta(days=1)
    origin_ctx = data_service.context(as_of=as_of)
    preds = scenario_predictor.predict(scenario_task, origin_ctx)
    scenario_payload = preds[0].metadata
    with open(SCENARIO_CACHE, "w") as f:
        json.dump(scenario_payload, f, indent=2)
    print("Saved scenario analysis.")

# ── Scenario cards ────────────────────────────────────────────────────────────
scenario_origin_price_row = price_df[price_df.index >= SCENARIO_AS_OF]
scenario_origin_price = (
    float(scenario_origin_price_row.iloc[0]["price"]) if not scenario_origin_price_row.empty else float("nan")
)

display(
    Markdown(
        f"#### Agent response — Stream 3  "
        f"*(origin: {SCENARIO_AS_OF.date()}, LNR ${scenario_origin_price:.2f}/share)*\n\n"
        f"Base case: **{scenario_payload.get('base_case', '?')}**"
    )
)

base_case = scenario_payload.get("base_case", "")
for s in scenario_payload.get("scenarios", []):
    name = s.get("name", "?")
    desc = s.get("description", "")
    prob = float(s.get("probability", 0))
    rng = s.get("lnr_range_60d", [float("nan"), float("nan")])
    lo_r, hi_r = float(rng[0]), float(rng[1])
    pe = float(s.get("point_estimate_60d", float("nan")))
    drivers = s.get("key_drivers", [])
    base_marker = "  ★ **base case**" if name == base_case else ""

    display(
        Markdown(
            f"---\n"
            f"**{name}**{base_marker}\n\n"
            f"{desc}\n\n"
            f"| | |\n|---|---|\n"
            f"| Probability | **{prob:.0%}**  `{prob_bar(prob)}` |\n"
            f"| LNR range (60 days) | ${lo_r:.0f} – ${hi_r:.0f} /share |\n"
            f"| Point estimate | **${pe:.0f} /share** |\n"
            f"| Key drivers | {' · '.join(drivers) if drivers else '—'} |\n"
        )
    )

overall = scenario_payload.get("rationale", "")
if overall:
    display(Markdown(f"---\n\n> **Overall reasoning:** {overall}"))

Saved scenario analysis.


#### Agent response — Stream 3  *(origin: 2026-02-02, LNR $85.57/share)*

Base case: **Steady Operational Outperformance**

---
**Steady Operational Outperformance**  ★ **base case**

Linamar maintains strong free cash flow and successfully integrates recent acquisitions, leading to steady margin expansion despite broader industrial sector weakness.

| | |
|---|---|
| Probability | **50%**  `█████░░░░░  50%` |
| LNR range (60 days) | $84 – $92 /share |
| Point estimate | **$88 /share** |
| Key drivers | Continued robust free cash flow generation · Successful integration of Aludyne and GF Leipzig assets |


---
**Industrial Cyclical Headwinds**

Prolonged weakness in the industrial segment, specifically Skyjack and agricultural equipment demand, offsets gains in the Mobility segment and forces a downward valuation revision.

| | |
|---|---|
| Probability | **30%**  `███░░░░░░░  30%` |
| LNR range (60 days) | $78 – $85 /share |
| Point estimate | **$82 /share** |
| Key drivers | Further decline in demand for industrial capital equipment · Rising geopolitical trade frictions impacting export margins |


---
**Strategic Acquisition Pivot**

Linamar announces a major new capital allocation move or strategic pivot to capitalize on their strong balance sheet, temporarily impacting margins but signaling long-term growth.

| | |
|---|---|
| Probability | **20%**  `██░░░░░░░░  20%` |
| LNR range (60 days) | $82 – $90 /share |
| Point estimate | **$86 /share** |
| Key drivers | Utilization of cash reserves for new M&A activity · Market sentiment responding to long-term diversification strategy |


---

> **Overall reasoning:** As of early 2026, Linamar has demonstrated a consistent ability to generate record cash flow and maintain disciplined operational performance, even when faced with industrial segment headwinds. The stock's recent price action, hovering near its highs around $85, reflects confidence in management's diversification strategy and their capacity to leverage a strong balance sheet for growth. While macroeconomic and industrial cyclical risks remain, the 'Steady Operational Outperformance' remains the most likely path as the market prioritizes earnings stability and disciplined capital allocation over speculative growth. Other scenarios are considered subordinate to the firm's demonstrated ability to navigate industrial cycles through its diversified mobility and industrial segments.

---

## Summary

**One identity, three roles.** The shared `analyst_config` (system instruction +
`search_web` toolbelt) never changes across streams. Each stream assigns a role
with an editable **task spec** in the user payload via `LnrMultitaskPromptBuilder`,
plus a stream-specific `output_schema`.

That is the bootcamp pattern for multi-task agentic forecasting. Notebooks 02/04
still use a trajectory-specialized system prompt (`build_lnr_news_config`) for
scored backtests — a useful contrast: bake the contract into identity, or keep
identity stable and swap the user-message ask.

Continue to [`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb)
for the stateless backtest harness, then Notebooks 5–6 for the adaptive agent
training and protected evaluation.
